one row per symbol + date + session_id Each row represents one unusual volume episode.

Source: coinwatch.silver.candles

Minute-level data is required, so this cannot use daily_metrics.

In [0]:
%sql

with flagged as (
    select
        symbol,
        open_time,
        date,
        volume,
        close,
        volume > 3 * avg(volume) over (
            partition by symbol
            order by open_time
            rows between 60 preceding and 1 preceding
        )as is_burst
    from coinwatch.silver.candles
),
sessions as (
    select
    *,
    sum(
        case
            when is_burst
                and not coalesce(
                    lag(is_burst) over (partition by symbol order by open_time),
                false)
            then 1
            else 0
        end
    ) over (partition by symbol order by open_time)
    as session_id
    from flagged
)

select
    symbol,
    date,
    session_id,
    min(open_time) as session_start,
    max(open_time) as session_end,
    count(*) as burst_minutes,
    sum(volume) as session_volume,
    max(volume) as peak_minute_volume
from sessions
where is_burst
group by
    symbol,
    date,
    session_id